# 5장 — 시간을 잘게 썰기: 오일러법과 시간 간격 (핵심)

웹 챕터(5장)와 **같은 예제, 같은 수치**를 파이썬으로 계산합니다:
오일러법, 간격 DT에 따른 매끄러움·가짜 출렁임·발산, DT를 반으로 줄일 때의 오차, 그리고 학습률 = DT.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch5')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 오일러법: 지금의 변화율로 DT만큼 나아가기

$$S(t + \Delta t) = S(t) + \Delta t \times f\big(S(t)\big)$$

균형 루프 하나를 계산합니다: 격차의 변화율 $= -\text{격차}/\tau$, τ = 4개월, 처음 격차 100. 정확한 답은 $100\,e^{-t/\tau}$.

In [ ]:
import math

TAU, S0 = 4, 100

def euler(f, s0, dt, t_end):
    # 웹 EulerLab 과 같은 계산: [(t, S), ...]
    out = [(0.0, s0)]
    s, t = s0, 0.0
    for _ in range(round(t_end / dt)):
        s = s + dt * f(s)
        t = t + dt
        out.append((t, s))
    return out

decay = lambda s: -s / TAU
exact = lambda t: S0 * math.exp(-t / TAU)

print(' DT   DT/τ   한 걸음 배수   처음 다섯 걸음')
for dt in [1, 2, 3, 4, 6, 8, 9, 10]:
    pts = euler(decay, S0, dt, 5 * dt)
    print(f'{dt:3d}   {dt / TAU:4.2f}   × {1 - dt / TAU:6.2f}     {[round(s, 1) for _, s in pts]}')

assert [round(s, 1) for _, s in euler(decay, S0, 6, 30)] == [100, -50, 25, -12.5, 6.2, -3.1]   # 가짜 출렁임
assert [s for _, s in euler(decay, S0, 8, 40)] == [100, -100, 100, -100, 100, -100]            # 경계: 영원히
assert [round(s, 2) for _, s in euler(decay, S0, 9, 27)] == [100, -125, 156.25, -195.31]         # 발산
assert [s for _, s in euler(decay, S0, 2, 4)] == [100, 50, 25]                                 # 계산 연습
print('✅ DT < τ 매끄럽게 · τ < DT < 2τ 가짜 출렁임 · DT = 2τ 경계 · DT > 2τ 발산')

## 2. 그림에서 식으로: 저량-유량도가 계산 규칙이 된다

오일러법 한 줄의 $f(S)$, 곧 '저량이 S일 때의 순유량'은 저량-유량도에서 그대로 읽어 냅니다(웹 5장 2절).

1. **상자마다 식이 하나**: 상자가 변하는 속도 = 들어오는 파이프 − 나가는 파이프.
2. **밸브마다 식이 하나**: 그 밸브로 들어오는 가는 화살표의 출발점들이 식의 재료. 화살표가 오지 않는 값은 그 식에 없다.
3. **작은 원은 숫자 하나**(또는 다른 값으로 만든 식).

4장 회의실: 상자 '실내 온도' ← 파이프 '데우는 속도' ← 화살표(목표 온도, τ, 실내 온도). 화살표는 재료만 알려 주고, 섞는 법(빼고 나누기)은
모형을 만드는 사람이 밸브에 적습니다. 그렇게 만든 식을 오일러법에 넣는 것이 시스템 다이내믹스 도구가 속에서 하는 계산이에요.

In [ ]:
# 4장 회의실 그림을 식으로 옮겨 오일러법으로 계산
GOAL, TAU_HEAT = 22, 10            # 작은 원: 목표 온도(°C), 조정 시간 τ(분)

def heating(temp):                 # 밸브 '데우는 속도': 화살표가 오는 세 값(목표, τ, 실내 온도)이 재료
    return (GOAL - temp) / TAU_HEAT

def d_temp(temp):                  # 상자 '실내 온도': 들어오는 파이프 − 나가는 파이프(나가는 파이프 없음)
    return heating(temp) - 0

temp, step = 12.0, 1 / 16
for _ in range(round(10 / step)):  # 10분, DT = 1/16분
    temp += d_temp(temp) * step
exact10 = GOAL - (GOAL - 12) * math.exp(-10 / TAU_HEAT)
print(f'10분 뒤 {temp:.2f}°C (정확한 답 {exact10:.2f}°C, 4장의 18.3°C)')
assert abs(temp - exact10) < 0.05
# 확인 문제: 들어오는 파이프 20·15명/월, 나가는 파이프 10명/월 → 상자의 변화 속도 25명/월
assert sum([20, 15]) - sum([10]) == 25
print('✅ 그림 → 식 → 오일러법')

## 3. DT를 반으로 줄이면 오차도 반으로 (1차 정확도)

t = 8개월(= 2τ)에서 정확한 답은 $100 e^{-2} \approx 13.53$입니다.

In [ ]:
t_check = 8
rows = []
for dt in [2, 1, 0.5, 0.25, 0.125]:
    v = euler(decay, S0, dt, t_check)[-1][1]
    rows.append((dt, v, abs(v - exact(t_check))))
    print(f'DT = {dt:5.3f}  걸음 {round(t_check / dt):3d}  오일러 {v:7.3f}   오차 {abs(v - exact(t_check)):6.3f}')

assert [round(e, 2) for _, _, e in rows] == [7.28, 3.52, 1.73, 0.85, 0.43]
for (_, _, e1), (_, _, e2) in zip(rows, rows[1:]):
    assert 1.9 < e1 / e2 < 2.2        # 반으로 줄면 오차도 거의 반
print('✅ 오차 7.28 → 3.52 → 1.73 → 0.85 → 0.43: 1차 정확도')

## 4. 더 똑똑한 방법: RK4 (룽게-쿠타 4차)

한 칸 안에서 기울기를 네 번 재 평균을 냅니다. DT = 1로 8개월을 계산해 오일러법과 비교해요.

In [ ]:
def rk4(f, s0, dt, t_end):
    s = s0
    for _ in range(round(t_end / dt)):
        k1 = f(s)
        k2 = f(s + dt / 2 * k1)
        k3 = f(s + dt / 2 * k2)
        k4 = f(s + dt * k3)
        s = s + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return s

e_euler = abs(euler(decay, S0, 1, 8)[-1][1] - exact(8))
e_rk4 = abs(rk4(decay, S0, 1, 8) - exact(8))
e_rk4_half = abs(rk4(decay, S0, 0.5, 8) - exact(8))
print(f'DT = 1: 오일러 오차 {e_euler:.4f},  RK4 오차 {e_rk4:.4f}')
print(f'RK4 에서 DT 를 반으로: 오차 {e_rk4:.6f} → {e_rk4_half:.6f} ({e_rk4 / e_rk4_half:.1f}분의 1)')
assert round(e_euler, 2) == 3.52 and e_rk4 < 0.002
assert 14 < e_rk4 / e_rk4_half < 18            # 4차 정확도: 약 1/16
print('✅ RK4 는 같은 DT 에서 오차가 수천 배 작고, DT 를 반으로 줄이면 오차가 약 1/16')

## 5. 학습률은 시간 간격이다

경사 흐름 $\frac{d\theta}{dt} = -\nabla L(\theta)$ 를 간격 $\eta$로 오일러법 계산하면 경사하강 $\theta_{k+1} = \theta_k - \eta \nabla L(\theta_k)$ 가 됩니다.
곡률 λ = 2인 골짜기 $L = \theta^2$ (기울기 $2\theta$)에서는 시간 상수 τ = 1/λ = 0.5, 그래서 DT/τ = ηλ.

In [ ]:
lam = 2
grad = lambda th: lam * th          # L = (λ/2)·θ² 의 기울기

def gradient_descent(eta, th0=8, steps=6):
    th, out = th0, [th0]
    for _ in range(steps):
        th = th - eta * grad(th)
        out.append(th)
    return out

for eta in [0.3, 0.8, 1.0, 1.2]:
    gd_path = gradient_descent(eta)
    flow_path = [s for _, s in euler(lambda th: -grad(th), 8, eta, 6 * eta)]
    assert all(abs(a - b) < 1e-9 for a, b in zip(gd_path, flow_path))   # 완전히 같은 계산
    print(f'학습률 {eta:3.1f}  (ηλ = DT/τ = {eta * lam:3.1f}):  {[round(v, 2) for v in gd_path]}')
print('✅ 경사하강 = 경사 흐름의 오일러법 · 발산 경계 ηλ < 2 = DT < 2τ (8부 2장)')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), sharey=True)
ts = [i * 0.1 for i in range(401)]
for ax, dt in zip(axes, [2, 6, 8, 9]):
    pts = euler(decay, S0, dt, 40)
    ax.plot(ts, [exact(t) for t in ts], '--', color='#64748b', label='정확한 답')
    ax.plot([t for t, _ in pts], [s for _, s in pts], marker='o', color='#4f46e5', label=f'오일러 DT = {dt}')
    ax.axhline(0, color='#94a3b8', lw=1)
    ax.set_ylim(-250, 250)
    ax.set_title(f'DT/τ = {dt / TAU:.2f}')
    ax.set_xlabel('개월')
    ax.legend(fontsize=8)
axes[0].set_ylabel('격차')
plt.tight_layout()
plt.show()

## 6. 오늘의 발견

- **그림에서 식으로**: 상자마다 '들어오는 − 나가는 파이프', 밸브마다 화살표가 오는 값으로 만든 식. 그 식을 오일러법에 넣는 것이 도구가 하는 계산이다.
- **오일러법**: 저량(다음) = 저량(지금) + 순유량 × DT. 시스템 다이내믹스 도구가 미래를 계산하는 기본 방법.
- 시간 상수 τ인 균형 루프에서 한 걸음 배수는 $1 - DT/\tau$: DT < τ 매끄럽게, τ < DT < 2τ 가짜 출렁임, DT > 2τ 발산.
- DT를 반으로 줄이면 오차도 반(1차 정확도). RK4는 1/16(4차 정확도).
- **학습률 = DT**: 경사하강은 경사 흐름의 오일러법이고, 8부 2장의 발산 경계(학습률 × 곡률 < 2)는 DT < 2τ와 같은 식.
- 출렁임이 진짜인지 확인하려면 DT를 반으로 줄여 다시 계산한다.

다음 장(6장 늦게 오는 소식)에서 시스템 자체가 출렁이는 이유, 지연을 봅니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**